In [1]:
%cd ../..

c:\Users\ajaoo\Desktop\Projects\hospitalization_research


In [2]:
import pandas as pd
import numpy as np
import os

import shutil
import joblib
import plotly.io as pio
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error as mae, mean_squared_error as mse
from src.forecasting.ml_forecasting import calculate_metrics
from src.utils import ts_utils
import plotly.express as px
import plotly.graph_objects as go
from itertools import cycle
import time
import warnings
from tqdm.notebook import tqdm
from pathlib import Path
np.random.seed(42)
tqdm.pandas()
from src.utils import plotting_utils


pio.templates.default = "plotly_white"

c:\Users\ajaoo\Desktop\Projects\hospitalization_research\src\utils\data_utils.py:6: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
os.makedirs("data/output", exist_ok=True)
preprocessed = Path("data/NHS_region/Timeseries")
output = Path("data/output")

In [4]:
def format_plot(
    fig, legends=None, xlabel="Time", ylabel="Value", title="", font_size=15
):
    if legends:
        names = cycle(legends)
        fig.for_each_trace(lambda t: t.update(name=next(names)))
    fig.update_layout(
        autosize=False,
        width=900,
        height=500,
        title_text=title,
        title={"x": 0.5, "xanchor": "center", "yanchor": "top"},
        titlefont={"size": 20},
        legend_title=None,
        legend=dict(
            font=dict(size=font_size),
            orientation="h",
            yanchor="bottom",
            y=0.98,
            xanchor="right",
            x=1,
        ),
        yaxis=dict(
            title_text=ylabel,
            titlefont=dict(size=font_size),
            tickfont=dict(size=font_size),
        ),
        xaxis=dict(
            title_text=xlabel,
            titlefont=dict(size=font_size),
            tickfont=dict(size=font_size),
        ),
    )
    return fig

In [5]:
def mase(actual, predicted, insample_actual):
    mae_insample = np.mean(np.abs(np.diff(insample_actual)))
    mae_outsample = np.mean(np.abs(actual - predicted))
    return mae_outsample / mae_insample


def forecast_bias(actual, predicted):
    return np.mean(predicted - actual)
def plot_forecast(pred_df, forecast_columns, forecast_display_names=None):
    if forecast_display_names is None:
        forecast_display_names = forecast_columns
    else:
        assert len(forecast_columns) == len(forecast_display_names)
    mask = ~pred_df[forecast_columns[0]].isnull()
    colors = [
        "rgba(" + ",".join([str(c) for c in plotting_utils.hex_to_rgb(c)]) + ",<alpha>)"
        for c in px.colors.qualitative.Plotly
    ]
    act_color = colors[0]
    colors = cycle(colors[1:])
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=pred_df[mask].index,
            y=pred_df[mask].covidOccupiedMVBeds_trend_diff,  # Fixed the column name here
            mode="lines",
            line=dict(color=act_color.replace("<alpha>", "0.9")),
            name="7 day-moving average MVBeds_trend",   # Fixed the name here
        )
    )
    for col, display_col in zip(forecast_columns, forecast_display_names):
        fig.add_trace(
            go.Scatter(
                x=pred_df[mask].index,
                y=pred_df.loc[mask, col],
                mode="lines",
                line=dict(dash="dot", color=next(colors).replace("<alpha>", "1")),
                name=display_col,
            )
        )
    return fig



def highlight_abs_min(s, props=""):
    return np.where(s == np.nanmin(np.abs(s.values)), props, "")


In [6]:
try:
    train_df = pd.read_csv(preprocessed / "featured_eng_train.csv")
    test_df = pd.read_csv(preprocessed / "featured_eng_test.csv")
    val_df = pd.read_csv(preprocessed / "featured_eng_val.csv")

except FileNotFoundError:
    print("File not found, please run the feature engineering notebook first")

In [7]:
# Filter data for London
# Filter data for London
sample_train_df = train_df.loc[
    train_df.areaName == "London",
    [
        "date",
        'covidOccupiedMVBeds', 'cumAdmissions',
       'hospitalCases', 'newAdmissions', 'covidOccupiedMVBeds_trend',
       'cumAdmissions_trend', 'hospitalCases_trend', 'newAdmissions_trend',
       'covidOccupiedMVBeds_trend_diff',
       'covidOccupiedMVBeds_trend_seasonal_diff', 'covidOccupiedMVBeds_lag_1',
       'covidOccupiedMVBeds_lag_7', 'covidOccupiedMVBeds_lag_14',
       'covidOccupiedMVBeds_lag_21', 'covidOccupiedMVBeds_rolling_7_mean',
       'covidOccupiedMVBeds_rolling_7_std', 'day_of_week', 'week_of_year',
       'Month', 'Quarter'
    ],
]
sample_test_df = test_df.loc[
    test_df.areaName == "London",
    [
        'date',
        'cumAdmissions',
        'hospitalCases', 
        'newAdmissions', 
       'cumAdmissions_trend',
       'hospitalCases_trend',
       'newAdmissions_trend',
       'covidOccupiedMVBeds_trend_diff',
    'covidOccupiedMVBeds_lag_1',
       'covidOccupiedMVBeds_lag_7', 'covidOccupiedMVBeds_lag_14',
       'covidOccupiedMVBeds_lag_21', 'covidOccupiedMVBeds_rolling_7_mean',
       'covidOccupiedMVBeds_rolling_7_std', 'day_of_week', 'week_of_year',
       'Month', 'Quarter'
    ],
]

sample_val_df = val_df.loc[
    val_df.areaName == "London",
    [
        'date',
        'cumAdmissions',
        'hospitalCases', 
        'newAdmissions', 
       'cumAdmissions_trend',
       'hospitalCases_trend',
       'newAdmissions_trend',
       'covidOccupiedMVBeds_trend_diff',
    'covidOccupiedMVBeds_lag_1',
       'covidOccupiedMVBeds_lag_7', 'covidOccupiedMVBeds_lag_14',
       'covidOccupiedMVBeds_lag_21', 'covidOccupiedMVBeds_rolling_7_mean',
       'covidOccupiedMVBeds_rolling_7_std', 'day_of_week', 'week_of_year',
       'Month', 'Quarter'
    ],
]

In [8]:
# Convert date to datetime and set as index
sample_train_df["date"] = pd.to_datetime(sample_train_df["date"])
sample_test_df["date"] = pd.to_datetime(sample_test_df["date"])
sample_val_df["date"] = pd.to_datetime(sample_val_df["date"])

sample_train_df.set_index("date", inplace=True)
sample_test_df.set_index("date", inplace=True)
sample_val_df.set_index("date", inplace=True)

In [9]:
from src.dl.dataloaders import TimeSeriesDataModule
from src.dl.models import SingleStepRNNConfig, SingleStepRNNModel,RNNConfig, Seq2SeqConfig, Seq2SeqModel
import pytorch_lightning as pl

import torch
# For reproduceability set a random seed
pl.seed_everything(42)

Seed set to 42


42

In [10]:
sample_train_df.head()

,covidOccupiedMVBeds,cumAdmissions,hospitalCases,newAdmissions,covidOccupiedMVBeds_trend,cumAdmissions_trend,hospitalCases_trend,newAdmissions_trend,covidOccupiedMVBeds_trend_diff,covidOccupiedMVBeds_trend_seasonal_diff,covidOccupiedMVBeds_lag_1,covidOccupiedMVBeds_lag_7,covidOccupiedMVBeds_lag_14,covidOccupiedMVBeds_lag_21,covidOccupiedMVBeds_rolling_7_mean,covidOccupiedMVBeds_rolling_7_std,day_of_week,week_of_year,Month,Quarter
date,,,,,,,,,,,,,,,,,,,,
2022-06-13,61,123251,1001,114,61.000000,123595.142857,1003.142857,114.857143,-1.142857,-16.000000,57.0,69.0,75.0,73.0,61.000000,4.472136,0,24,6,2
2022-06-12,60,123137,967,88,60.000000,123480.285714,991.714286,109.428571,-1.000000,-14.571429,61.0,67.0,84.0,74.0,60.000000,3.605551,6,23,6,2
2022-06-11,60,123049,942,83,59.000000,123370.857143,981.571429,106.285714,-1.000000,-13.571429,60.0,67.0,81.0,72.0,59.000000,1.914854,5,23,6,2
2022-06-10,54,122966,932,93,58.571429,123264.571429,971.714286,104.285714,-0.428571,-10.285714,60.0,57.0,83.0,70.0,58.571429,2.636737,4,23,6,2
2022-06-09,51,122873,914,76,57.714286,123160.285714,960.142857,98.714286,-0.857143,-8.428571,54.0,57.0,76.0,70.0,57.714286,3.903600,3,23,6,2


In [11]:
target = "covidOccupiedMVBeds_trend_diff"
index_cols = ["date", "areaName"]
pred_df = pd.concat([sample_train_df[[target]], sample_test_df[[target]]])

In [12]:
sample_train_df['type'] = "train"
sample_val_df['type'] = "val"
sample_test_df['type'] = "test"
sample_df = pd.concat([sample_train_df[[target, "type"]], sample_val_df[[target, "type"]], sample_test_df[[target, "type"]],])
sample_df.head()

,covidOccupiedMVBeds_trend_diff,type
date,,
2022-06-13,-1.142857,train
2022-06-12,-1.000000,train
2022-06-11,-1.000000,train
2022-06-10,-0.428571,train
2022-06-09,-0.857143,train


In [13]:
sample_df['covidOccupiedMVBeds_trend_diff'] = sample_df['covidOccupiedMVBeds_trend_diff'].astype('float32')

In [14]:
HORIZON = 1
WINDOW = 7

In [15]:
datamodule = TimeSeriesDataModule(data = sample_df[[target]],
        n_val = sample_val_df.shape[0],
        n_test = sample_test_df.shape[0],
        window = WINDOW, # giving enough memory to capture daily seasonality
        horizon = HORIZON, # single step
        normalize = "global", # normalizing the data
        batch_size = 64,
        num_workers = 0)
datamodule.setup()

### LSTM-FC Seq2Seq

In [16]:
encoder_config = RNNConfig(
    input_size=1,
    hidden_size=128,
    num_layers=2,
    bidirectional=True,
).__dict__
rnn2fc_config = Seq2SeqConfig(
    encoder_type="LSTM",
    decoder_type="FC",
    encoder_params=encoder_config,
    decoder_params={"window_size": WINDOW, "horizon":HORIZON},
    decoder_use_all_hidden=False,
    learning_rate=1e-3,
)

model = Seq2SeqModel(rnn2fc_config)

trainer = pl.Trainer(
    min_epochs=5,
    max_epochs=100,
    callbacks=[pl.callbacks.EarlyStopping(monitor="valid_loss", patience=5)],
)
trainer.fit(model, datamodule)
# Removing artifacts created during training
shutil.rmtree("lightning_logs")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3060 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type    | Params
------------------------------------
0 | encoder | LSTM    | 529 K 
1 | decoder | Linear  | 257   
2 | loss    | MSELoss | 0     
------------------------------------
529 K     Trainable params
0         Non-trainable params
529 K     Total params
2.119     Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\ajaoo\miniconda3\envs\modern_ts_2E\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
c:\Users\ajaoo\miniconda3\envs\modern_ts_2E\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
c:\Users\ajaoo\miniconda3\envs\modern_ts_2E\Lib\site-packages\pytorch_lightning\loops\fit_loop.py:298: The number of training batches (13) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [17]:
tag = f"{rnn2fc_config.encoder_type}_{rnn2fc_config.decoder_type}_{'all_hidden' if rnn2fc_config.decoder_use_all_hidden else 'last_hidden'}"
pred = trainer.predict(model, datamodule.test_dataloader())
# pred is a list of outputs, one for each batch
pred = torch.cat(pred).squeeze().detach().numpy()
# Apply reverse transformation because we applied global normalization
pred = pred * datamodule.train.std + datamodule.train.mean
pred_df_ = pd.DataFrame({tag: pred}, index=sample_test_df.index)
pred_df = pred_df.join(pred_df_)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\ajaoo\miniconda3\envs\modern_ts_2E\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

In [18]:
metric_record = []
actuals = sample_test_df[target].values
algorithm_name = tag

metrics = {
    "Algorithm": algorithm_name,
    "MAE": mae(actuals, pred),
    "MSE": mse(actuals, pred),
    "MASE": mase(actuals, pred, sample_train_df[target].values),
    "Forecast Bias": forecast_bias(actuals, pred),
}

value_formats = ["{}", "{:.4f}", "{:.4f}", "{:.4f}", "{:.2f}"] # Added a format for the Algorithm name
metrics = {key: format_.format(value) for key, value, format_ in zip(metrics.keys(), metrics.values(), value_formats)}
metric_record.append(metrics)
print(metrics)

{'Algorithm': 'LSTM_FC_last_hidden', 'MAE': '0.8112', 'MSE': '1.1638', 'MASE': '0.5657', 'Forecast Bias': '-0.42'}


In [19]:
# Plotting the forecast
fig = plot_forecast(pred_df, forecast_columns=[tag], forecast_display_names=[tag])
title = f"{rnn2fc_config.encoder_type}: MAE: {metrics['MAE']} | MSE: {metrics['MSE']} | MASE: {metrics['MASE']} | Bias: {metrics['Forecast Bias']}"
fig = format_plot(fig, title=title)
fig.update_xaxes(type="date", range=["2022-03-01", "2022-05-01"])
fig.show()

### LSTM-FC Seq2Seq use all hidden

In [20]:
encoder_config = RNNConfig(
    input_size=1,
    hidden_size=128,
    num_layers=3,
    bidirectional=True,
).__dict__
rnn2fc_config = Seq2SeqConfig(
    encoder_type="LSTM",
    decoder_type="FC",
    encoder_params=encoder_config,
    decoder_params={"window_size": WINDOW, "horizon":HORIZON},
    decoder_use_all_hidden=True,
    learning_rate=1e-3,
)

model = Seq2SeqModel(rnn2fc_config)

trainer = pl.Trainer(
    min_epochs=5,
    max_epochs=100,
    callbacks=[pl.callbacks.EarlyStopping(monitor="valid_loss", patience=5)],
)
trainer.fit(model, datamodule)
# Removing artifacts created during training
shutil.rmtree("lightning_logs")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type    | Params
------------------------------------
0 | encoder | LSTM    | 924 K 
1 | decoder | Linear  | 1.8 K 
2 | loss    | MSELoss | 0     
------------------------------------
926 K     Trainable params
0         Non-trainable params
926 K     Total params
3.706     Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\ajaoo\miniconda3\envs\modern_ts_2E\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: PossibleUserWarning:

The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.

c:\Users\ajaoo\miniconda3\envs\modern_ts_2E\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: PossibleUserWarning:

The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.

c:\Users\ajaoo\miniconda3\envs\modern_ts_2E\Lib\site-packages\pytorch_lightning\loops\fit_loop.py:298: PossibleUserWarning:

The number of training batches (13) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the trainin

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [21]:
tag = f"{rnn2fc_config.encoder_type}_{rnn2fc_config.decoder_type}_{'all_hidden' if rnn2fc_config.decoder_use_all_hidden else 'last_hidden'}"
pred = trainer.predict(model, datamodule.test_dataloader())
# pred is a list of outputs, one for each batch
pred = torch.cat(pred).squeeze().detach().numpy()
# Apply reverse transformation because we applied global normalization
pred = pred * datamodule.train.std + datamodule.train.mean
pred_df_ = pd.DataFrame({tag: pred}, index=sample_test_df.index)
pred_df = pred_df.join(pred_df_)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\ajaoo\miniconda3\envs\modern_ts_2E\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: PossibleUserWarning:

The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.



Predicting: |          | 0/? [00:00<?, ?it/s]

In [22]:
actuals = sample_test_df[target].values
algorithm_name = tag

metrics = {
    "Algorithm": algorithm_name,
    "MAE": mae(actuals, pred),
    "MSE": mse(actuals, pred),
    "MASE": mase(actuals, pred, sample_train_df[target].values),
    "Forecast Bias": forecast_bias(actuals, pred),
}

value_formats = ["{}", "{:.4f}", "{:.4f}", "{:.4f}", "{:.2f}"] # Added a format for the Algorithm name
metrics = {key: format_.format(value) for key, value, format_ in zip(metrics.keys(), metrics.values(), value_formats)}
metric_record.append(metrics)
print(metrics)

{'Algorithm': 'LSTM_FC_all_hidden', 'MAE': '0.8452', 'MSE': '1.2688', 'MASE': '0.5895', 'Forecast Bias': '-0.45'}


In [23]:
# Plotting the forecast
fig = plot_forecast(pred_df, forecast_columns=[tag], forecast_display_names=[tag])
title = f"{rnn2fc_config.encoder_type}: MAE: {metrics['MAE']} | MSE: {metrics['MSE']} | MASE: {metrics['MASE']} | Bias: {metrics['Forecast Bias']}"
fig = format_plot(fig, title=title)
fig.update_xaxes(type="date", range=["2022-03-01", "2022-05-01"])
fig.show()

In [24]:
encoder_config = RNNConfig(
    input_size=1,
    hidden_size=128,
    num_layers=3,
    bidirectional=True,
).__dict__
rnn2rnn_config = Seq2SeqConfig(
    encoder_type="LSTM",
    decoder_type="LSTM",
    encoder_params=encoder_config,
    decoder_params=encoder_config,
    learning_rate=1e-3,
)

model = Seq2SeqModel(rnn2rnn_config)

trainer = pl.Trainer(
    min_epochs=5,
    max_epochs=100,
    callbacks=[pl.callbacks.EarlyStopping(monitor="valid_loss", patience=5)],
)
trainer.fit(model, datamodule)
# Removing artifacts created during training
shutil.rmtree("lightning_logs")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type    | Params
------------------------------------
0 | encoder | LSTM    | 924 K 
1 | decoder | LSTM    | 924 K 
2 | fc      | Linear  | 257   
3 | loss    | MSELoss | 0     
------------------------------------
1.8 M     Trainable params
0         Non-trainable params
1.8 M     Total params
7.398     Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\ajaoo\miniconda3\envs\modern_ts_2E\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: PossibleUserWarning:

The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.

c:\Users\ajaoo\miniconda3\envs\modern_ts_2E\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: PossibleUserWarning:

The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.

c:\Users\ajaoo\miniconda3\envs\modern_ts_2E\Lib\site-packages\pytorch_lightning\loops\fit_loop.py:298: PossibleUserWarning:

The number of training batches (13) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the trainin

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [25]:
tag = f"{rnn2rnn_config.encoder_type}_{rnn2rnn_config.decoder_type}"
pred = trainer.predict(model, datamodule.test_dataloader())
# pred is a list of outputs, one for each batch
pred = torch.cat(pred).squeeze().detach().numpy()
# Apply reverse transformation because we applied global normalization
pred = pred * datamodule.train.std + datamodule.train.mean
pred_df_ = pd.DataFrame({tag: pred}, index=sample_test_df.index)
pred_df = pred_df.join(pred_df_)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\ajaoo\miniconda3\envs\modern_ts_2E\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: PossibleUserWarning:

The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.



Predicting: |          | 0/? [00:00<?, ?it/s]

In [26]:
actuals = sample_test_df[target].values
algorithm_name = tag

metrics = {
    "Algorithm": algorithm_name,
    "MAE": mae(actuals, pred),
    "MSE": mse(actuals, pred),
    "MASE": mase(actuals, pred, sample_train_df[target].values),
    "Forecast Bias": forecast_bias(actuals, pred),
}

value_formats = ["{}", "{:.4f}", "{:.4f}", "{:.4f}", "{:.2f}"] # Added a format for the Algorithm name
metrics = {key: format_.format(value) for key, value, format_ in zip(metrics.keys(), metrics.values(), value_formats)}
metric_record.append(metrics)
print(metrics)

{'Algorithm': 'LSTM_LSTM', 'MAE': '0.7707', 'MSE': '1.1357', 'MASE': '0.5375', 'Forecast Bias': '-0.08'}


In [27]:
# Plotting the forecast
fig = plot_forecast(pred_df, forecast_columns=[tag], forecast_display_names=[tag])
title = f"{rnn2fc_config.encoder_type}: MAE: {metrics['MAE']} | MSE: {metrics['MSE']} | MASE: {metrics['MASE']} | Bias: {metrics['Forecast Bias']}"
fig = format_plot(fig, title=title)
fig.update_xaxes(type="date", range=["2022-03-01", "2022-05-01"])
fig.show()

In [28]:
metric_df = pd.DataFrame(metric_record)
metric_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Algorithm      3 non-null      object
 1   MAE            3 non-null      object
 2   MSE            3 non-null      object
 3   MASE           3 non-null      object
 4   Forecast Bias  3 non-null      object
dtypes: object(5)
memory usage: 248.0+ bytes


In [29]:
metric_df[["MAE", "MSE", "MASE", "Forecast Bias"]] = metric_df[["MAE", "MSE", "MASE", "Forecast Bias"]].astype('float32')


In [30]:
formatted = metric_df.style.format({
    "MAE": "{:.4f}",
    "MSE": "{:.4f}",
    "MASE": "{:.4f}",
    "Forecast Bias": "{:.2f}%",
    "Time Elapsed": "{:.6f}",
})
formatted = formatted.highlight_min(
    color="lightgreen", subset=["MAE", "MSE", "MASE"]
).apply(
    highlight_abs_min,
    props="color:black;background-color:lightgreen",
    axis=0,
    subset=["Forecast Bias"],
)
formatted


,Algorithm,MAE,MSE,MASE,Forecast Bias
0,LSTM_FC_last_hidden,0.8112,1.1638,0.5657,-0.42%
1,LSTM_FC_all_hidden,0.8452,1.2688,0.5895,-0.45%
2,LSTM_LSTM,0.7707,1.1357,0.5375,-0.08%
